## Daily booking revenue (baseline, Prompt 1b)
What Genie Code produced in the dry run **without** instructions or skills. Kept as the comparison for 2a/2b.

Writes `<catalog>.<schema>.booking_revenue_daily`: one row per booking date, from `samples.wanderbricks.bookings`.

Gaps against the team standard (the checklist in `DEMO_SCRIPT.md`, segment 1):
* No layer prefix (`booking_revenue_daily`, not `gold_booking_revenue_daily`)
* No table or column COMMENTs, no TBLPROPERTIES or tags
* No `audit_timestamp` / `source_system` columns
* Cancelled bookings are **included** in revenue
* Job (see the job screenshot): no failure notification, timeout, retries, tags or max concurrent runs

In [ ]:
dbutils.widgets.text("catalog", "uaigw_fe")
dbutils.widgets.text("schema", "asml_dry_run")

catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("schema")
target = f"{catalog}.{schema}.booking_revenue_daily"

In [ ]:
from pyspark.sql import functions as F

df = (
    spark.table("samples.wanderbricks.bookings")
    .groupBy(F.to_date("created_at").alias("booking_date"))
    .agg(
        F.count("*").alias("bookings"),
        F.round(F.sum("total_amount"), 2).alias("revenue"),
        F.round(F.avg("total_amount"), 2).alias("avg_booking_value"),
    )
    .orderBy("booking_date")
)

df.write.mode("overwrite").saveAsTable(target)
display(spark.table(target).orderBy(F.desc("booking_date")).limit(10))